In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import os

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),  # 若走原生 OpenAI，可传 None/不传
    temperature=0.2,  # 稳定输出
    timeout=1200,  # 超时保护（秒）
    max_retries=2,  # 简单重试
)

In [3]:
from langchain_core.tools import tool


@tool
def multiply(a: int, b: int) -> int:
    """将两个整数相乘。"""
    return a * b


@tool
def add(a: int, b: int) -> int:
    """将两个整数相加。"""
    return a + b


# 创建工具列表
tools = [multiply, add]

In [4]:
llm_with_tools = llm.bind_tools(tools)

In [5]:
# 提一个需要使用工具的问题
query = "3乘以12等于多少？另外，11加上49等于多少？"
response = llm_with_tools.invoke(query)

print(response)
# 查看模型生成的工具调用
print(response.tool_calls)
# 输出:
# [{'name': 'multiply', 'args': {'a': 3, 'b': 12}, 'id': '...'},
#  {'name': 'add', 'args': {'a': 11, 'b': 49}, 'id': '...'}]

content='<think>\n好的，用户问了两个问题，一个是3乘以12等于多少，另一个是11加上49等于多少。我需要先确定应该使用哪个工具来处理这两个问题。\n\n首先看第一个问题，3乘以12。根据提供的工具，有一个multiply函数，它的参数是两个整数a和b，所以这里应该调用multiply函数，参数a是3，b是12。\n\n然后是第二个问题，11加上49。这里应该使用add函数，同样参数a是11，b是49。需要确保这两个函数调用都正确生成。\n\n接下来要检查参数是否都是整数，用户给出的数字都是整数，所以没问题。然后按照要求，每个函数调用都要放在tool_call标签里，返回JSON对象。所以需要生成两个tool_call部分，分别对应multiply和add的调用。\n\n最后，确保JSON的格式正确，键名和结构符合给定的函数定义。没有其他工具需要调用，所以只需要这两个函数即可。用户可能需要两个结果，所以需要分别处理，不能遗漏任何一个。\n</think>\n\n' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 286, 'prompt_tokens': 244, 'total_tokens': 530, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'Qwen/Qwen3-32B', 'system_fingerprint': None, 'id': 'chatcmpl-dba467dbae4c4e89a7d9c018dbeb1da9', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019dc3e5-c9bd-7631-b437-8082623adadd-0' tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 12}, 'id': 'chatcmpl-tool-0bfd3891c61f419b9213bf687